# 05 — Statistical Models — Euronext EMA

**Adapté de** `main/05_statistical_models.ipynb` — version Euronext.

**Cible** : Euronext EMA (EUR/tonne) au lieu du CBOT (USD/bu).

**Features** : CBOT existantes + données EU + cross-market (basis, EUR/USD).

Ce notebook reproduit la même structure que le notebook CBOT mais avec :
- Cible `y_up_h40_ema` / `y_logret_h40_ema`
- Features augmentées (basis CBOT-EMA, eurusd_zscore_52w)
- Comparaison directe CBOT vs EMA sur les mêmes splits walk-forward

**Pré-requis** : Avoir complété `01_ema_data_collection.ipynb`

In [ ]:
import sys
from pathlib import Path

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root / "src"))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Chargement features CBOT + données EMA

In [ ]:
from mais.paths import FEATURES_PARQUET, TARGETS_PARQUET, RAW_DIR
from mais.utils import read_parquet

features = read_parquet(FEATURES_PARQUET)
targets = read_parquet(TARGETS_PARQUET)

print(f"Features CBOT: {features.shape}")
print(f"Targets CBOT: {targets.shape}")
print(f"\nExemples cibles: {[c for c in targets.columns if 'h40' in c][:5]}")

In [ ]:
# Charger les prix EMA
ema_path = RAW_DIR / "euronext_ema" / "euronext_ema.csv"
eurusd_path = RAW_DIR / "eu_cross_assets" / "eu_cross_assets.csv"

ema_prices = None
eurusd = None

if ema_path.exists():
    ema_prices = pd.read_csv(ema_path, parse_dates=["Date"]).set_index("Date")
    print(f"EMA: {ema_prices.shape} — {ema_prices.index.min().date()} → {ema_prices.index.max().date()}")
else:
    print("⚠ EMA non disponible — lancer 01_ema_data_collection.ipynb d'abord")

if eurusd_path.exists():
    eurusd = pd.read_csv(eurusd_path, parse_dates=["Date"]).set_index("Date")
    print(f"EUR/USD: {eurusd.shape}")

## 2. Construction cibles EMA + features cross-market

In [ ]:
def build_ema_targets(ema_prices, horizons=(20, 40, 60)):
    if ema_prices is None:
        return pd.DataFrame()
    close = ema_prices["ema_close"].dropna()
    result = {}
    for h in horizons:
        fwd = close.shift(-h)
        result[f"y_logret_h{h}_ema"] = np.log(fwd / close)
        result[f"y_up_h{h}_ema"] = (fwd > close).astype(int)
    return pd.DataFrame(result)


def build_cross_features(ema_prices, eurusd, features):
    """Construire les features cross-market: basis CBOT-EMA, eurusd z-score."""
    cross = pd.DataFrame(index=features.index)

    if eurusd is not None and "eurusd_rate" in eurusd.columns:
        eur = eurusd["eurusd_rate"].reindex(features.index, method="ffill")
        # Expanding z-score anti-leakage
        cross["eurusd_zscore_52w"] = (
            (eur - eur.expanding().mean()) / eur.expanding().std()
        ).shift(1)  # shift(1) = valeur d'hier

        # Basis CBOT (USD/bu → EUR/t) - EMA (EUR/t)
        # Chercher la colonne close CBOT dans les features
        cbot_close = next((features[c] for c in features.columns if "corn_close" in c or "cbot_close" in c), None)
        if cbot_close is not None and ema_prices is not None:
            ema_close = ema_prices["ema_close"].reindex(features.index, method="ffill")
            cbot_eur_t = cbot_close * 39.368 / eur
            basis = cbot_eur_t - ema_close
            basis_shift = basis.shift(1)  # anti-leakage
            basis_mean = basis_shift.expanding().mean()
            basis_std = basis_shift.expanding().std()
            cross["cbot_ema_basis_zscore"] = (basis_shift - basis_mean) / basis_std

    return cross


ema_targets = build_ema_targets(ema_prices)
cross_features = build_cross_features(ema_prices, eurusd, features)

print(f"Cibles EMA: {ema_targets.shape}")
print(f"Features cross-market: {cross_features.shape}, colonnes: {list(cross_features.columns)}")

## 3. Comparaison modèles CBOT vs EMA — Walk-forward OOF

In [ ]:
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


MODELS = {
    "ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RidgeClassifier(alpha=10.0, class_weight="balanced")),
    ]),
    "logistic": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
    ]),
}


def walk_forward_oof(X: pd.DataFrame, y: pd.Series, model_name="ridge",
                     n_splits=8, min_train_years=3, n_bootstrap=500) -> dict:
    import copy
    common = X.index.intersection(y.dropna().index)
    X_ = X.loc[common].fillna(X.loc[common].median())
    y_ = y.loc[common]

    n = len(X_)
    min_train = 252 * min_train_years
    step = max((n - min_train) // n_splits, 21)

    preds = np.full(n, np.nan)
    split_das = []
    model = copy.deepcopy(MODELS[model_name])

    for i in range(n_splits):
        tr_end = min_train + i * step
        te_end = min(tr_end + step, n)
        if tr_end >= n:
            break
        X_tr, y_tr = X_.iloc[:tr_end], y_.iloc[:tr_end]
        X_te, y_te = X_.iloc[tr_end:te_end], y_.iloc[tr_end:te_end]
        if len(y_te) < 5 or y_tr.nunique() < 2:
            continue
        model.fit(X_tr, y_tr)
        p = model.predict(X_te)
        preds[tr_end:te_end] = p
        split_das.append((p == y_te.values).mean())

    valid = ~np.isnan(preds)
    if not valid.any():
        return {"da": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan, "n": 0}

    da = (preds[valid] == y_.values[valid]).mean()
    rng = np.random.default_rng(42)
    yv = y_.values[valid]; pv = preds[valid]
    boot = [(pv[rng.integers(0, len(yv), len(yv))] == yv[rng.integers(0, len(yv), len(yv))]).mean()
            for _ in range(n_bootstrap)]

    return {
        "da": da,
        "ci95_lo": np.percentile(boot, 2.5),
        "ci95_hi": np.percentile(boot, 97.5),
        "n": int(valid.sum()),
        "annual_std": np.std(split_das),
        "split_das": split_das,
    }


print("Fonctions définies.")

In [ ]:
# Lancer le benchmark sur h40
H = 40
results = []

# Features CBOT seules
X_cbot = features.drop(columns=["Date"], errors="ignore")
X_cbot_cross = pd.concat([X_cbot, cross_features], axis=1)

for model_name in ["ridge"]:
    # CBOT target
    cbot_col = next((c for c in targets.columns if f"h{H}" in c and "up" in c.lower()), None)
    if cbot_col:
        r = walk_forward_oof(X_cbot, targets[cbot_col], model_name=model_name)
        r.update({"target": "CBOT", "features": "cbot_only", "horizon": H, "model": model_name})
        results.append(r)
        print(f"CBOT h{H} ({model_name}): DA={r['da']:.4f} [{r['ci95_lo']:.4f}, {r['ci95_hi']:.4f}]")

    # EMA target — features CBOT seules
    if not ema_targets.empty:
        ema_col = f"y_up_h{H}_ema"
        if ema_col in ema_targets.columns:
            r = walk_forward_oof(X_cbot, ema_targets[ema_col], model_name=model_name)
            r.update({"target": "EMA", "features": "cbot_only", "horizon": H, "model": model_name})
            results.append(r)
            print(f"EMA  h{H} ({model_name}, cbot_only): DA={r['da']:.4f} [{r['ci95_lo']:.4f}, {r['ci95_hi']:.4f}]")

            # EMA target — features CBOT + cross-market
            if not cross_features.empty:
                r2 = walk_forward_oof(X_cbot_cross, ema_targets[ema_col], model_name=model_name)
                r2.update({"target": "EMA", "features": "cbot+cross", "horizon": H, "model": model_name})
                results.append(r2)
                print(f"EMA  h{H} ({model_name}, cbot+cross): DA={r2['da']:.4f} [{r2['ci95_lo']:.4f}, {r2['ci95_hi']:.4f}]")
    else:
        print("⚠ Cibles EMA non disponibles")

In [ ]:
if results:
    df_res = pd.DataFrame(results)[["target", "features", "horizon", "model", "da", "ci95_lo", "ci95_hi", "n", "annual_std"]]
    print("\n" + "=" * 80)
    print("RÉSULTATS BENCHMARK EMA vs CBOT")
    print("=" * 80)
    print(df_res.to_string(index=False, float_format="{:.4f}".format))

    # Sauvegarder
    out_dir = project_root / "artefacts" / "benchmark_ema_vs_cbot"
    out_dir.mkdir(parents=True, exist_ok=True)
    df_res.to_csv(out_dir / f"benchmark_h{H}.csv", index=False)
    print(f"\nSauvegardé: {out_dir / f'benchmark_h{H}.csv'}")
else:
    print("⚠ Aucun résultat")

## 4. Analyse des erreurs

Quels types de jours le modèle prédit mal sur EMA ?

In [ ]:
# Analyse des erreurs spécifique EMA
# À compléter une fois les données EMA disponibles
# Structure identique à main/05 mais sur la cible EMA
print("Section à développer une fois les données EMA disponibles.")
print("Voir main/05_statistical_models.ipynb pour la structure complète.")

## 5. Conclusion

Ce notebook adapte l'analyse CBOT vers la cible Euronext EMA.

**Prochaines étapes** :
- `06_ml_models_ema.ipynb` — LightGBM, XGBoost sur target EMA
- `00_benchmark_pivot_ema.ipynb` — Décision finale pivot
- `10_module_a_dashboard.ipynb` — Indicateur contexte marché EU